# MNIST Autoencoder — Analysis

Training now happens in `train.py`:

    uv run python projects/mnist/autoencoder/train.py

This notebook only loads the resulting checkpoint for analysis:
training curves, reconstruction quality, and translation invariance /
equivariance.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import (
    peak_signal_noise_ratio,
    structural_similarity_index_measure,
)
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from chimera.data import MNISTDataModule
from chimera.models import DigitDreamerAE

RUN_DIR = "/mnt/ai/runs/mnist/autoencoder"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VARIANT = "tiny"

## Load checkpoint

In [ ]:
ckpt = torch.load(f"{RUN_DIR}/checkpoints/ae.ckpt", map_location="cpu", weights_only=False)
model_state = {
    k.removeprefix("model."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = DigitDreamerAE.from_variant(VARIANT, in_channels=1, latent_dim=1)
model.load_state_dict(model_state)
model.to(DEVICE).eval()
print(f"loaded AE ({sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt['epoch']})")

In [ ]:
dm = MNISTDataModule(data_dir=DATA_DIR, pin_memory=False, num_workers=4, image_size=32)
dm.prepare_data()
dm.setup("test")
test_loader = dm.test_dataloader()

## Training curves

In [ ]:
csv_path = sorted(glob.glob(f"{RUN_DIR}/csv/version_*/metrics.csv"))[-1]
metrics = pd.read_csv(csv_path)

plt.figure(figsize=(7, 5))
plt.title("Reconstruction Loss")
for stage in ["train", "val"]:
    col = f"{stage}/loss"
    d = metrics.dropna(subset=[col])
    plt.plot(d["step"], d[col], marker="o" if stage == "val" else None, label=stage)
plt.xlabel("Step")
plt.ylabel("MAE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Reconstruction quality

In [ ]:
@torch.no_grad()
def recon_metrics(model, loader, device):
    maes, psnrs, ssims = [], [], []
    for x, _ in loader:
        x = x.to(device)
        recon = model(x)
        maes.append(F.l1_loss(recon, x).item())
        psnrs.append(peak_signal_noise_ratio(recon, x, data_range=1.0).item())
        ssims.append(structural_similarity_index_measure(recon, x, data_range=1.0).item())
    return np.mean(maes), np.mean(psnrs), np.mean(ssims)


mae, psnr, ssim = recon_metrics(model, test_loader, DEVICE)
print(f"test MAE {mae:.4f}, PSNR {psnr:.2f} dB, SSIM {ssim:.3f}")

images, _ = next(iter(test_loader))
with torch.no_grad():
    recons = model(images.to(DEVICE)).float().cpu()

n = 8
batch = torch.cat([images[:n], recons[:n]]).clamp(0, 1)
grid = make_grid(batch, nrow=n)
plt.figure(figsize=(12, 3))
plt.imshow(to_pil_image(grid), cmap="gray")
plt.title("Original (top) vs Reconstruction (bottom)")
plt.axis("off")
plt.show()

## Translation invariance / equivariance

`DigitDreamerAE` is a fully-convolutional DC-AE; an equivariant AE would
satisfy `recon(shift(x)) == shift(recon(x))`. Measure how far off that is,
and how reconstruction quality of a shifted digit degrades.

In [ ]:
def shift_batch(x, dx, dy):
    """Shift (B, C, H, W) by dx (right) / dy (down) pixels, zero-filled."""
    out = torch.zeros_like(x)
    H, W = x.shape[-2:]
    sy, sx = slice(max(0, -dy), H - max(0, dy)), slice(max(0, -dx), W - max(0, dx))
    dys, dxs = slice(max(0, dy), H - max(0, -dy)), slice(max(0, dx), W - max(0, -dx))
    out[..., dys, dxs] = x[..., sy, sx]
    return out


@torch.no_grad()
def ae_shift_sweep(model, loader, shifts, device):
    x_all = torch.cat([x for x, _ in loader]).to(device)
    base_recon = model(x_all)
    base_mae = F.l1_loss(base_recon, x_all).item()
    base_psnr = peak_signal_noise_ratio(base_recon, x_all, data_range=1.0).item()

    recon_psnr = np.zeros((len(shifts), len(shifts)))
    equiv_mae = np.zeros((len(shifts), len(shifts)))
    for i, dy in enumerate(shifts):
        for j, dx in enumerate(shifts):
            xs = shift_batch(x_all, dx, dy)
            recon_xs = model(xs)
            recon_psnr[i, j] = peak_signal_noise_ratio(recon_xs, xs, data_range=1.0).item()
            equiv_mae[i, j] = F.l1_loss(recon_xs, shift_batch(base_recon, dx, dy)).item()
    return (base_mae, base_psnr), recon_psnr, equiv_mae


shifts = list(range(-8, 9, 1))
center = shifts.index(0)
(base_mae, base_psnr), recon_psnr, equiv_mae = ae_shift_sweep(model, test_loader, shifts, DEVICE)

print(f"baseline recon: MAE {base_mae:.4f}, PSNR {base_psnr:.2f} dB")
for s in [1, 2, 3, 4, 6, 8]:
    print(
        f"  shift +{s}px: recon PSNR {recon_psnr[center, center + s]:.2f} dB, "
        f"equivariance MAE {equiv_mae[center, center + s]:.4f}"
    )

extent = [shifts[0] - 0.5, shifts[-1] + 0.5, shifts[-1] + 0.5, shifts[0] - 0.5]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"DigitDreamerAE translation — baseline PSNR {base_psnr:.2f} dB")
im0 = axes[0].imshow(recon_psnr, cmap="viridis", extent=extent)
axes[0].set_title("Recon PSNR of shifted digit (dB)")
axes[0].set_xlabel("dx")
axes[0].set_ylabel("dy")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(equiv_mae, cmap="magma", extent=extent)
axes[1].set_title("Equivariance error MAE\n|recon(shift(x)) - shift(recon(x))|")
axes[1].set_xlabel("dx")
axes[1].set_ylabel("dy")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].plot(shifts, recon_psnr[center], marker="o", c="tab:green")
axes[2].axhline(base_psnr, ls="--", c="gray", alpha=0.6, label="baseline")
axes[2].set_title("Recon PSNR vs horizontal shift")
axes[2].set_xlabel("dx (px)")
axes[2].set_ylabel("PSNR (dB)")
axes[2].legend()
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()